### Prepare environment

In [0]:
%run ../environment/prepare_environment

# 5.3 Termination Policies

## Introduction
Training machine learning models can be time-consuming and expensive. "Termination policies" refer to strategies to stop the training or tuning process early if it's not yielding good results.

We will explore two types of early termination:
1.  **Manual Early Stopping**: Implementing a custom training loop with `SGDClassifier` to demonstrate a custom termination policy.
2.  **Early Stopping in XGBoost**: Using built-in callbacks to stop when validation performance plateaus.
3.  **Pruning in Hyperparameter Tuning**: Stopping a specific trial in a hyperparameter search (e.g., Optuna) if intermediate results are poor compared to other trials.

In [0]:
import xgboost as xgb
import optuna
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import SGDClassifier
import matplotlib.pyplot as plt
from copy import deepcopy

# Prepare data
data = load_breast_cancer()

# Split: Train (60%), Validation (20%), Test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42) # 0.25 * 0.8 = 0.2

print(f"Train size: {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

## 1. Manual Early Stopping Implementation

To understand how termination policies work, we will implement a custom training loop using `SGDClassifier` (Logistic Regression). We will monitor the validation accuracy and stop if it doesn't improve based on our defined policy.

**Policy Parameters:**
*   `delay`: Warmup period (epochs) before starting checks.
*   `evaluation_factor`: Minimum percentage improvement required to reset the counter.
*   `evaluation_interval`: Number of consecutive runs (patience) allowed without improvement.
*   `last_best_model`: We will restore the best model found during training.

In [0]:
def train_with_early_stopping(X_train, y_train, X_val, y_val, 
                              max_epochs=100, 
                              delay=5, 
                              evaluation_factor=0.001, 
                              evaluation_interval=5):
    
    # Initialize model (SGDClassifier allows partial_fit for iterative training)
    model = SGDClassifier(loss='log_loss', learning_rate='constant', eta0=0.01, random_state=42)
    classes = np.unique(y_train)
    
    best_val_acc = 0.0
    best_model = None
    consecutive_no_improv = 0
    
    print(f"{'Epoch':<6} | {'Train Acc':<10} | {'Val Acc':<10} | {'Status'}")
    print("-" * 55)
    
    for epoch in range(max_epochs):
        # Train for one epoch
        model.partial_fit(X_train, y_train, classes=classes)
        
        # Evaluate
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        
        status = ""
        
        # Policy Logic
        if epoch < delay:
            status = "Warmup"
        else:
            # Check if new metric is better than best by evaluation_factor %
            # Threshold = best * (1 + factor)
            threshold = best_val_acc * (1 + evaluation_factor)
            
            if val_acc > threshold:
                best_val_acc = val_acc
                best_model = deepcopy(model)
                consecutive_no_improvement = 0
                status = "Improved"
            else:
                consecutive_no_improvement += 1
                status = f"No Improvement ({consecutive_no_improvement}/{evaluation_interval})"
        
            print(f"{epoch+1:<6} | {train_acc:.4f}     | {val_acc:.4f}     | {status}")
            
            # Termination Check
            if consecutive_no_improvement >= evaluation_interval:
                print(f"\nEarly stopping triggered! Restoring best model (Acc: {best_val_acc:.4f}).")
                model = best_model
                break

    mlflow.sklearn.log_model(
        model,
        name='SGDClassifier_EarlyStopping',
        input_example=X_train[:5]
    )
    
    return model

# Run the manual training
with mlflow.start_run(run_name="manual_early_stopping"):
    final_model = train_with_early_stopping(
        X_train, y_train, X_val, y_val,
        max_epochs=50,
        delay=5,                # Warmup
        evaluation_factor=0.001, # 0.1% improvement required
        evaluation_interval=5    # Patience
    )
    
    # Final Test Evaluation
    test_acc = accuracy_score(y_test, final_model.predict(X_test))
    mlflow.log_metric("test_acc", test_acc)
    print(f"\nFinal Test Set Accuracy: {test_acc:.4f}")

## 2. Early Stopping in XGBoost

When training gradient boosting models, we add trees iteratively. If adding more trees doesn't improve the validation score, we should stop to prevent overfitting and save time.

We use `early_stopping_rounds` parameter for this.

In [0]:
# Define model with a high number of estimators
model = xgb.XGBClassifier(
    n_estimators=1000, 
    learning_rate=0.05,
    early_stopping_rounds=20,  # Stop if no improvement for 20 rounds
    eval_metric="error"        # Metric to monitor
)

with mlflow.start_run(run_name="xgboost_early_stopping"):
    # Log parameters
    mlflow.log_params(model.get_params())
    
    # Fit model with eval_set
    print("Training with Early Stopping...")
    model.fit(
        X_train, y_train, 
        eval_set=[(X_train, y_train), (X_val, y_val)], 
        verbose=True
    )
    
    print(f"\nBest iteration: {model.best_iteration}")
    print(f"Best score: {model.best_score}")
    
    # Log metrics
    mlflow.log_metric("best_score", model.best_score)
    mlflow.log_metric("best_iteration", model.best_iteration)
    
    # Final evaluation on Test set
    test_preds = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)
    mlflow.log_metric("test_accuracy", test_acc)
    print(f"Test Accuracy: {test_acc}")
    
    results = model.evals_result()

    fig = plt.figure(figsize=(10,7))
    plt.plot(results["validation_0"]["error"], label="Training loss")
    plt.plot(results["validation_1"]["error"], label="Validation loss")
    plt.axvline(model.best_iteration, color="gray", label="Optimal tree number")
    plt.xlabel("Number of trees")
    plt.ylabel("Loss")
    plt.legend()

    mlflow.log_figure(fig, "loss_curve.png")

    # Log model
    mlflow.xgboost.log_model(
        model, 
        name = "xgboost_early_stopping",
        input_example=X_train[:5]
    )

## 3. Pruning in Optuna

When running hundreds of trials to find the best hyperparameters, some trials are obviously bad from the start (e.g., very high learning rate causing divergence). Optuna can "prune" (terminate) these trials early.

We need to report intermediate values to Optuna inside the objective function.

In [0]:
def objective_with_pruning(trial):
    param = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'booster': 'gbtree',
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'eval_metric': 'error'
    }

    # We use the lower-level xgb.train API to access iteration callbacks easily
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    # Define pruning callback
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-error")

    bst = xgb.train(
        param, 
        dtrain, 
        num_boost_round=100,
        evals=[(dval, "validation")],
        callbacks=[pruning_callback],
        verbose_eval=False
    )
    
    # Return the last evaluation result
    preds = bst.predict(dval)
    pred_labels = [round(value) for value in preds]
    accuracy = accuracy_score(y_val, pred_labels)
    return 1.0 - accuracy # Return error to minimize

We use `MedianPruner`, which prunes a trial if its intermediate result is worse than the median of intermediate results of previous trials at the same step.

In [0]:
# Define MLflow callback for Optuna
mlflow_callback = optuna.integration.MLflowCallback(
    metric_name="error",
    mlflow_kwargs={"nested": True} ,
    create_experiment=False
)

# Create study with a Pruner
study = optuna.create_study(
    direction='maximize', 
    study_name='optuna_pruning_study',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)

print("Starting optimization with pruning...")
with mlflow.start_run(run_name="optuna_pruning_study") as run:
    study.optimize(objective_with_pruning, n_trials=30, callbacks=[mlflow_callback])

pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print(f"\nStudy statistics: ")
print(f"  Number of finished trials: {len(study.trials)}")
print(f"  Number of pruned trials: {len(pruned_trials)}")
print(f"  Number of complete trials: {len(complete_trials)}")